# Buzzer Classification — Local End-to-End Collector

One notebook, top to bottom, TikTok and YouTube, on your own machine.

**What "end-to-end" means here, honestly:** YouTube is a pure API call — that cell runs with
zero intervention. TikTok requires a real login and real scrolling; there is no way to script
around that without violating the platform's terms (credential automation, fingerprint spoofing,
CAPTCHA bypass), and this notebook does not do that. Instead, that cell **pauses and waits for
you** — a Chromium window opens, you log in and/or scroll, you press Enter in the notebook,
execution continues automatically from there. Run the whole notebook with "Run All"; it will
simply stop and wait for you exactly where a human is genuinely required, then proceed on its own.

Login is a one-time cost. The browser profile persists to disk (`browser_profile/`), so the
second time you run this notebook you likely won't be prompted to log in again — only to confirm
the session still looks valid.

## Setup (once)

```bash
pip install playwright google-api-python-client pandas langdetect
playwright install chromium
```

Set one environment variable before launching Jupyter (or paste it into §1 — see the warning
there about not committing it):

```bash
export YOUTUBE_API_KEY="..."
```

## Edit before running

Section 2 is the only cell you need to change: video IDs and the TikTok post URLs you want to
collect comments from. Everything after that runs unattended except the browser pause, and writes
CSV files each time you run it.

## 1. Environment

In [1]:
import os, re, json, time, hashlib, asyncio, unicodedata
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
from collections import Counter
from typing import Any, Iterable

import pandas as pd
import numpy as np

COLLECTOR_VERSION = "local-e2e-1.0.0"

ROOT      = Path("./buzzer_data")
RAW       = ROOT / "raw"
CANONICAL = ROOT / "canonical"
FEATURES  = ROOT / "features"
PROFILE   = ROOT / "browser_profile"
for p in (RAW, CANONICAL, FEATURES, PROFILE):
    p.mkdir(parents=True, exist_ok=True)

YOUTUBE_API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not YOUTUBE_API_KEY:
    raise RuntimeError("YOUTUBE_API_KEY is not set — export it before launching Jupyter (see Setup above)")

def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()

def jsonl_append(path: Path, rows: Iterable[dict]) -> int:
    n = 0
    with path.open("a", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False, default=str) + "\n"); n += 1
    return n

def jsonl_read(path: Path) -> list[dict]:
    if not path.exists(): return []
    with path.open(encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]

print("collector", COLLECTOR_VERSION)
print("YouTube key: set")

collector local-e2e-1.0.0
YouTube key: set


## 2. What to collect — edit this cell

`BROWSER_TARGETS` entries need a `platform` (currently only `tiktok` is supported) and the direct
URL to the post whose comments you want. `max_scroll_rounds` controls how far the automated scroll
goes before handing control back to you.

In [2]:
#@ ---- YouTube ----
YT_VIDEO_IDS         = ["gksiY4Zxb54"]                             # e.g. ["dQw4w9WgXcQ"]
YT_MAX_PER_VIDEO     = 500     # capped per video — breadth (more videos) beats depth (one huge one)

#@ ---- TikTok ----
# Strategy: many same-topic videos, each capped, rather than one video scraped to exhaustion —
# coordination features only fire when the same accounts recur across posts, so breadth matters
# more than draining any single video's comment section.
# max_comments: PRIMARY stop — cap unique comments captured per video (~300-800 is a good range).
# target_coverage: fallback only, so a video smaller than the cap still finishes early.
# batch_rounds/max_batches: scrolls in batches of batch_rounds, up to max_batches batches,
#   checking real coverage after each. Total ceiling is roughly max_batches * (batch_rounds + ~30).
# pause_ms is a ceiling, not a fixed wait — each round polls for new content and moves on as
#   soon as it arrives, so most rounds finish well under this.
BROWSER_TARGETS = [
    {"platform": "tiktok",
     "url": "https://www.tiktok.com/@olivervisualfx/video/7657440258673413396?is_from_webapp=1&sender_device=pc",
     "max_comments": 500, "target_coverage": 0.95, "batch_rounds": 60, "max_batches": 8,
     "idle_limit": 12, "pause_ms": 1800, "expand_replies": True},
]

print(f"YouTube: {len(YT_VIDEO_IDS)} videos")
print(f"Browser: {len(BROWSER_TARGETS)} targets across "
      f"{sorted(set(t['platform'] for t in BROWSER_TARGETS)) or 'none'}")

YouTube: 1 videos
Browser: 1 targets across ['tiktok']


## 3. Canonical schema and text normalisation — shared by every platform

In [3]:
CANONICAL_FIELDS = [
    "user_id", "username", "display_name", "account_created_at", "is_verified",
    "has_custom_avatar", "bio_text", "location",
    "followers_count", "following_count", "total_posts_count",
    "post_id", "thread_id", "created_at", "text_content", "clean_text", "language",
    "hashtags", "user_mentions", "urls", "media_types", "source_device",
    "like_count", "repost_count", "reply_count", "is_repost", "is_reply",
    "user_recent_posts", "user_recent_timestamps",
    "_platform", "_collected_at", "_collector_version", "_source_url", "_raw_ref",
]
LIST_FIELDS = {"hashtags", "user_mentions", "urls", "media_types",
               "user_recent_posts", "user_recent_timestamps"}

def empty_record(**kw) -> dict:
    rec = {f: ([] if f in LIST_FIELDS else None) for f in CANONICAL_FIELDS}
    rec["_collector_version"] = COLLECTOR_VERSION
    rec["_collected_at"] = now_utc()
    rec.update(kw)
    return rec

def to_frame(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(records)
    for f in CANONICAL_FIELDS:
        if f not in df.columns:
            df[f] = [[] for _ in range(len(df))] if f in LIST_FIELDS else None
    return df[CANONICAL_FIELDS]

RE_URL     = re.compile(r"https?://\S+|www\.\S+", re.I)
RE_HASHTAG = re.compile("(?<!\\w)#([\\wÀ-ɏ؀-ۿ]+)", re.U)
RE_MENTION = re.compile(r"(?<!\w)@([\w.]+)", re.U)
RE_WS      = re.compile(r"\s+")
RE_ZWJ     = re.compile("[​-‏‪-‮⁠﻿]")
RE_RT      = re.compile(r"^RT @[\w]+:\s*")

def extract_entities(text: str) -> dict:
    t = text or ""
    return {"hashtags":      [h.lower() for h in RE_HASHTAG.findall(t)],
            "user_mentions": [m.lower().rstrip(".") for m in RE_MENTION.findall(t)],
            "urls":          RE_URL.findall(t)}

def clean_text(text: str) -> str:
    t = unicodedata.normalize("NFKC", text or "")
    t = RE_ZWJ.sub("", t); t = RE_RT.sub("", t)
    t = RE_URL.sub(" <url> ", t)
    t = RE_MENTION.sub(" <user> ", t)
    t = RE_HASHTAG.sub(lambda m: " " + m.group(1).lower() + " ", t)
    return RE_WS.sub(" ", t).strip()

def dedupe_key(text) -> str:
    # `text or ""` alone isn't enough: this is also called on text_content AFTER a CSV
    # round-trip (derive_features), where pandas turns an empty string back into a float
    # NaN on read — and `nan or ""` evaluates to `nan` (NaN is truthy), so `text` stays a
    # float and RE_RT.sub() crashes with "expected string or bytes-like object". Guard with
    # an explicit isinstance check instead.
    t = unicodedata.normalize("NFKD", RE_RT.sub("", text if isinstance(text, str) else "").casefold())
    t = "".join(c for c in t if c.isalnum())
    return hashlib.sha1(t.encode()).hexdigest() if t else ""

try:
    from langdetect import detect, DetectorFactory, LangDetectException
    DetectorFactory.seed = 0
    def detect_language(t):
        t = (t or "").strip()
        if len(t) < 12: return None
        try: return detect(t)
        except LangDetectException: return None
except ImportError:
    def detect_language(t): return None

def enrich_text_fields(rec: dict) -> dict:
    txt = rec.get("text_content") or ""
    if not rec.get("hashtags"): rec.update(extract_entities(txt))
    rec["clean_text"] = clean_text(txt)
    if not rec.get("language"): rec["language"] = detect_language(txt)
    return rec

def write_csv_with_lists(df: pd.DataFrame, path: Path) -> None:
    """CSV can't hold Python lists natively, so list-type columns are JSON-encoded on the way
    out. Always pair with read_csv_with_lists (or load_canonical) to get them back as lists
    rather than string reprs — otherwise every list-based feature silently reads as empty."""
    out = df.copy()
    for f in LIST_FIELDS:
        if f in out.columns:
            out[f] = out[f].map(lambda v: json.dumps(v if isinstance(v, list) else []))
    out.to_csv(path, index=False)

def read_csv_with_lists(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    df = pd.read_csv(path)
    for f in LIST_FIELDS:
        if f in df.columns:
            df[f] = df[f].map(lambda v: json.loads(v) if isinstance(v, str) else [])
    return df

def save_canonical(df: pd.DataFrame, platform: str) -> Path:
    path = CANONICAL / f"{platform}.csv"
    write_csv_with_lists(df, path)
    return path

def load_canonical(platform: str) -> pd.DataFrame | None:
    return read_csv_with_lists(CANONICAL / f"{platform}.csv")

print(len(CANONICAL_FIELDS), "canonical fields ready")

34 canonical fields ready


## 4. YouTube — runs unattended

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

def yt_fetch(video_id, max_comments):
    yt, out, token = build("youtube","v3",developerKey=YOUTUBE_API_KEY,cache_discovery=False), [], None
    while len(out) < max_comments:
        try:
            r = yt.commentThreads().list(part="snippet,replies", videoId=video_id, maxResults=100,
                    pageToken=token, textFormat="plainText", order="time").execute()
        except HttpError as e:
            print(f"  ! {video_id}: {e.reason}"); break
        for item in r.get("items", []):
            out.append({"_kind":"top","_video_id":video_id,"item":item})
            for rep in item.get("replies", {}).get("comments", []):
                out.append({"_kind":"reply","_video_id":video_id,"_parent":item["id"],"item":rep})
        token = r.get("nextPageToken")
        if not token: break
        time.sleep(0.1)
    return out[:max_comments]

def yt_normalise(raw):
    it = raw["item"]
    sn = it["snippet"].get("topLevelComment", it)["snippet"] if raw["_kind"]=="top" else it["snippet"]
    rec = empty_record(
        _platform="youtube", _raw_ref=it["id"],
        _source_url=f"https://www.youtube.com/watch?v={raw['_video_id']}",
        user_id=(sn.get("authorChannelId") or {}).get("value"),
        display_name=sn.get("authorDisplayName"),
        post_id=it["id"], thread_id=raw.get("_parent") or it["id"],
        created_at=sn.get("publishedAt"),
        text_content=sn.get("textOriginal") or sn.get("textDisplay") or "",
        like_count=sn.get("likeCount"),
        reply_count=it["snippet"].get("totalReplyCount") if raw["_kind"]=="top" else 0,
        is_reply=raw["_kind"]=="reply", is_repost=False,
        repost_count=None, source_device=None, media_types=["text"])
    return enrich_text_fields(rec)

def yt_enrich_channels(channel_ids):
    yt, out = build("youtube","v3",developerKey=YOUTUBE_API_KEY,cache_discovery=False), {}
    ids = [c for c in dict.fromkeys(channel_ids) if c]
    for i in range(0, len(ids), 50):
        try:
            r = yt.channels().list(part="snippet,statistics", id=",".join(ids[i:i+50])).execute()
        except HttpError as e:
            print(f"  ! channels {i}: {e.reason}"); continue
        for ch in r.get("items", []):
            sn, st = ch.get("snippet",{}), ch.get("statistics",{})
            thumb = (sn.get("thumbnails",{}).get("default",{}) or {}).get("url","")
            out[ch["id"]] = {
                "username": (sn.get("customUrl") or "").lstrip("@") or None,
                "account_created_at": sn.get("publishedAt"),
                "bio_text": sn.get("description"), "location": sn.get("country"),
                "followers_count": int(st["subscriberCount"]) if not st.get("hiddenSubscriberCount")
                                   and "subscriberCount" in st else None,
                "total_posts_count": int(st["videoCount"]) if "videoCount" in st else None,
                "has_custom_avatar": ("/ytc/default" not in thumb) if thumb else None,
            }
        time.sleep(0.1)
    return out

def run_youtube_collection():
    if not YOUTUBE_API_KEY:
        print("YOUTUBE_API_KEY not set — skipping YouTube"); return None
    if not YT_VIDEO_IDS:
        print("no YT_VIDEO_IDS configured — skipping YouTube"); return None
    recs = []
    for vid in YT_VIDEO_IDS:
        raw = yt_fetch(vid, YT_MAX_PER_VIDEO)
        jsonl_append(RAW / "youtube_raw.jsonl", raw)
        recs += [yt_normalise(r) for r in raw]
        print(f"  {vid}: {len(raw)} comments")
    if not recs:
        print("youtube: nothing collected"); return None
    prof = yt_enrich_channels([r["user_id"] for r in recs])
    for r in recs:
        r.update({k:v for k,v in prof.get(r["user_id"], {}).items() if v is not None})
    df = to_frame(recs).drop_duplicates(subset=["post_id"], keep="last")
    save_canonical(df, "youtube")
    print(f"youtube: {len(df)} comments, {df.user_id.nunique()} accounts")
    return df

df_yt = run_youtube_collection()

## 5. TikTok — the part that pauses for you

`Harvester` opens a real, visible Chromium window and records the JSON that the comment section
actually loads over the network, rather than parsing the DOM. The window uses a persistent
profile (`browser_profile/tiktok/`), so a login you complete once is still there next time you
run this notebook.

**When you run the next cell:** a browser opens. Log in if asked. Then come back to the notebook
and press **Enter in the input box that appears below the cell** — execution resumes automatically
and scrolls the comment section for you. This is the one genuinely manual step in the whole
notebook, and it is manual because making it automatic would mean automating a login, which is
where "scraping public data" turns into "circumventing platform security controls."

In [ ]:
# Sync Playwright API, driven from a dedicated worker thread.
#
# Two separate Windows/Jupyter issues stack here, and both need fixing:
#
# 1. ipykernel keeps an asyncio loop running in the main thread for its own use (zmq comms),
#    and Playwright's sync API refuses to run inside a thread that already has a loop
#    running ("Sync API inside the asyncio loop" error). Fix: call it from a fresh thread
#    that has no loop of its own — the ThreadPoolExecutor below.
# 2. That's not sufficient on Windows. ipykernel deliberately sets the *global* asyncio
#    event loop policy to WindowsSelectorEventLoopPolicy at startup, because pyzmq (which
#    the kernel's comms depend on) doesn't support ProactorEventLoop. That policy is
#    process-wide, not thread-local, so even a brand-new thread inherits it — and
#    Playwright's driver process needs Proactor to be spawned at all, hence the
#    NotImplementedError. Fix: explicitly switch the policy to Proactor before Playwright
#    starts. This is safe: it only changes what future new_event_loop() calls hand out —
#    the kernel's own loop object was already created at startup and keeps running as-is,
#    unaffected by a later policy change.
import sys, asyncio
if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

from playwright.sync_api import sync_playwright
from concurrent.futures import ThreadPoolExecutor

COMMENT_ENDPOINTS = {
    "tiktok": ["/api/comment/list", "/api/comment/reply"],
}

# Populated by run_browser_collection: platform -> unique comments captured THIS session
# (before raw JSONL accumulation from earlier runs). §6 uses this to explain any gap between
# "what this run scrolled" and "what's in the canonical file" (which includes older runs too).
SESSION_SEEN: dict[str, int] = {}

class Harvester:
    def __init__(self, platform: str):
        self.platform, self.patterns = platform, COMMENT_ENDPOINTS[platform]
        self.captured, self.ctx, self.pw, self.page = [], None, None, None

    def start(self):
        self.pw  = sync_playwright().start()
        self.ctx = self.pw.chromium.launch_persistent_context(
            user_data_dir=str(PROFILE / self.platform), headless=False,
            viewport={"width": 1440, "height": 900},
            locale="id-ID", timezone_id="Asia/Jakarta",
            args=["--disable-blink-features=AutomationControlled"],
        )
        self.page = self.ctx.pages[0] if self.ctx.pages else self.ctx.new_page()
        self.page.on("response", self._on_response)
        return self

    def _on_response(self, resp):
        if not any(p in resp.url for p in self.patterns): return
        ct = (resp.headers.get("content-type") or "")
        if "json" not in ct and "text" not in ct: return
        try: body = resp.text()
        except Exception: return
        try: payload = json.loads(body)
        except json.JSONDecodeError:
            cleaned = body.split("\n")[0].lstrip("for (;;);")
            try: payload = json.loads(cleaned)
            except Exception: return
        self.captured.append({"_platform": self.platform, "_url": resp.url,
            "_page_url": self.page.url, "_captured_at": now_utc(), "payload": payload})

    def open(self, url: str):
        self.page.goto(url, wait_until="domcontentloaded", timeout=60_000)
        self.page.wait_for_timeout(3000)

    def _hover_comment_panel(self):
        """
        page.mouse.wheel() fires at the CURRENT mouse position, which defaults to (0, 0) —
        the page's top-left corner — until something moves it. TikTok's comment list is a
        scrollable panel on the right side of the video page, not the whole document, so a
        wheel event at (0, 0) mostly misses it entirely. Hovering over the panel first (with
        a viewport-relative fallback if TikTok's DOM/testid has changed) fixes that.
        """
        try:
            panel = self.page.locator('[data-e2e="comment-list"]').first
            panel.wait_for(state="attached", timeout=3000)
            box = panel.bounding_box()
            if box:
                self.page.mouse.move(box["x"] + box["width"] / 2, box["y"] + box["height"] / 2)
                return
        except Exception:
            pass
        vp = self.page.viewport_size or {"width": 1440, "height": 900}
        self.page.mouse.move(vp["width"] * 0.75, vp["height"] / 2)

    def _wait_for_growth(self, timeout_ms: int, poll_ms: int = 150):
        """
        Waits up to timeout_ms for a new payload to arrive, but returns the moment one does
        instead of always sleeping the full window. Genuinely idle rounds (nothing new
        loading) still pay the full timeout_ms — that's necessary to distinguish "slow" from
        "actually plateaued".
        """
        start = len(self.captured)
        elapsed = 0
        while elapsed < timeout_ms:
            self.page.wait_for_timeout(poll_ms)
            elapsed += poll_ms
            if len(self.captured) > start:
                return True
        return False

    def scroll_comments(self, max_rounds: int = 200, idle_limit: int = 8, pause_ms: int = 1800):
        """
        Scrolls until capture growth stalls for `idle_limit` consecutive rounds, or
        `max_rounds` is hit — not a fixed count. TikTok lazy-loads ~20 comments per XHR.
        """
        self._hover_comment_panel()
        stagnant, last = 0, len(self.captured)
        for i in range(max_rounds):
            self.page.mouse.wheel(0, 3200)
            self._wait_for_growth(pause_ms)
            if len(self.captured) == last:
                stagnant += 1
                if stagnant >= idle_limit:
                    print(f"    plateaued after round {i} ({len(self.captured)} payloads captured)")
                    break
            else:
                stagnant, last = 0, len(self.captured)
            if (i + 1) % 10 == 0:
                print(f"    round {i+1}/{max_rounds}: {len(self.captured)} payloads captured")
        return len(self.captured)

    def expand_replies(self, max_clicks: int = 300, pause_ms: int = 900):
        """
        Clicks every visible 'view replies' button so nested replies load through the same
        /api/comment/reply endpoint the harvester already listens for. Best-effort: TikTok's
        button text/selectors vary by locale, so this may need adjusting if it clicks nothing.
        Each click waits only until its reply payload actually arrives (see _wait_for_growth).
        """
        clicked = 0
        for _ in range(max_clicks):
            buttons = self.page.locator("text=/repl(y|ies)|balasan/i")
            if buttons.count() == 0:
                break
            try:
                buttons.first.click(timeout=3000)
                clicked += 1
                self._wait_for_growth(pause_ms)
            except Exception:
                break
        if clicked:
            print(f"    expanded {clicked} reply thread(s)")
        return clicked

    def _coverage_snapshot(self):
        """Real coverage from what's actually been captured so far, vs. TikTok's own
        reported total — usable mid-run, before flush() writes anything to disk."""
        seen_ids = {c.get("cid") for row in self.captured
                    for c in (row["payload"].get("comments") or []) if c.get("cid")}
        totals = [row["payload"].get("total") for row in self.captured
                  if row["payload"].get("total")]
        return len(seen_ids), (max(totals) if totals else None)

    def scroll_until_coverage(self, target_pct: float = 0.95, batch_rounds: int = 60,
                               max_batches: int = 8, idle_limit: int = 12,
                               pause_ms: int = 1800, expand_replies: bool = True,
                               max_comments: int | None = 500):
        """
        Scrolls in batches instead of one blind pass, checking real coverage (unique
        comments captured vs. TikTok's own reported total) after each batch.

        `max_comments` is the PRIMARY stop condition now — the strategy is many videos with a
        capped sample each (breadth, for cross-video account/coordination signal), not one
        video scraped to exhaustion. `target_pct` only matters as a fallback for videos smaller
        than the cap (it lets those finish early rather than idling through empty batches).
        Also stops on a genuine plateau (two consecutive batches with zero new unique comments
        — TikTok itself has stopped serving more via scroll) or when `max_batches` is
        exhausted. Replies are expanded every batch since new 'view replies' buttons appear as
        more top-level comments load in. Always prints which of these happened, so a short run
        is diagnosable without re-reading the whole log.
        """
        prev_seen = -1
        stop_reason = "max_batches exhausted"
        for batch in range(max_batches):
            self.scroll_comments(max_rounds=batch_rounds, idle_limit=idle_limit, pause_ms=pause_ms)
            if expand_replies:
                self.expand_replies(pause_ms=max(pause_ms // 2, 600))
                self.scroll_comments(max_rounds=30, idle_limit=6, pause_ms=pause_ms)
            seen, total = self._coverage_snapshot()
            pct_str = f"{seen/total:.1%}" if total else "total not seen yet"
            print(f"    batch {batch+1}/{max_batches}: {seen} comments ({pct_str})"
                  + (f" — cap {max_comments}" if max_comments else ""))
            if max_comments and seen >= max_comments:
                stop_reason = f"reached max_comments cap ({max_comments})"
                break
            if total and seen / total >= target_pct:
                stop_reason = f"reached target coverage ({target_pct:.0%}) — video smaller than cap"
                break
            if seen == prev_seen:
                stop_reason = "plateaued — two batches in a row with no new comments"
                break
            prev_seen = seen
        seen, total = self._coverage_snapshot()
        print(f"    stopped: {stop_reason}")
        return seen, total

    def flush(self) -> int:
        n = jsonl_append(RAW / f"{self.platform}_payloads.jsonl", self.captured)
        self.captured = []
        return n

    def close(self):
        if self.ctx: self.ctx.close()
        if self.pw:  self.pw.stop()


def run_browser_collection(targets: list[dict]):
    """
    Groups targets by platform so login happens at most once per platform per run.
    Every Harvester call is submitted to the same single-worker thread and .result()
    is awaited immediately, so from the notebook's point of view this still runs top
    to bottom in order — the threading is invisible except that it fixes the Windows
    subprocess issue. input() prompts happen on the main thread as normal and block
    exactly as long as it takes you to log in and get ready; that's expected, not a hang.
    """
    if not targets:
        print("no BROWSER_TARGETS configured — skipping"); return

    by_platform: dict[str, list[dict]] = {}
    for t in targets:
        by_platform.setdefault(t["platform"], []).append(t)

    with ThreadPoolExecutor(max_workers=1) as ex:
        for platform, items in by_platform.items():
            print(f"\n=== {platform} ({len(items)} target(s)) ===")
            h = ex.submit(lambda p=platform: Harvester(p).start()).result()
            ex.submit(h.open, items[0]["url"]).result()
            input(f"  [{platform}] Log in if prompted, then press Enter here to continue... ")

            for i, t in enumerate(items):
                if i > 0:
                    ex.submit(h.open, t["url"]).result()
                print(f"  scrolling: {t['url']}")
                seen, total = ex.submit(h.scroll_until_coverage,
                                         t.get("target_coverage", 0.95),
                                         t.get("batch_rounds", 60),
                                         t.get("max_batches", 8),
                                         t.get("idle_limit", 12),
                                         t.get("pause_ms", 1800),
                                         t.get("expand_replies", True),
                                         t.get("max_comments", 500)).result()
                pct = f"{seen/total:.1%}" if total else "unknown"
                print(f"  final: {seen}/{total or '?'} comments captured ({pct})")

            # snapshot BEFORE flush() clears h.captured — this is "unique comments this
            # session actually captured for this platform", used by §6 to explain any gap
            # between that and the canonical file's total (which includes earlier runs too,
            # since raw JSONL accumulates and is never auto-cleared).
            SESSION_SEEN[platform], _ = ex.submit(h._coverage_snapshot).result()
            saved = ex.submit(h.flush).result()
            ex.submit(h.close).result()
            print(f"  {platform}: {saved} payloads written to raw/{platform}_payloads.jsonl")

run_browser_collection(BROWSER_TARGETS)

## 6. Parse the captured browser payloads

In [5]:
def dig(obj, *paths, default=None):
    for path in paths:
        cur, ok = obj, True
        for key in path.split("."):
            if isinstance(cur, dict) and key in cur: cur = cur[key]
            elif isinstance(cur, list) and key.isdigit() and int(key) < len(cur): cur = cur[int(key)]
            else: ok = False; break
        if ok and cur is not None: return cur
    return default

def parse_tiktok(row):
    out = []
    for c in dig(row["payload"], "comments", default=[]) or []:
        u = c.get("user", {}) or {}
        reply_id = c.get("reply_id")
        is_reply = bool(reply_id and reply_id != "0")
        # thread_id groups a top-level comment with all of its replies so coordination and
        # arrival-gap features mean the same thing on TikTok as on YouTube: a top-level
        # comment roots its own thread (thread_id = its own cid); a reply joins its parent's
        # thread (thread_id = reply_id = parent cid). The earlier version set top-level
        # comments to aweme_id (the whole video), which made every commenter on a video share
        # one giant "thread" — that overran coordination's max_group cap (silently zeroing
        # thread_co_* for TikTok) and made thread_id non-comparable across platforms.
        thread_id = reply_id if is_reply else c.get("cid")
        out.append(enrich_text_fields(empty_record(
            _platform="tiktok", _source_url=row.get("_page_url"), _raw_ref=c.get("cid"),
            user_id=u.get("sec_uid") or u.get("uid"), username=u.get("unique_id"),
            display_name=u.get("nickname"), bio_text=u.get("signature"),
            is_verified=bool(dig(u,"custom_verify","enterprise_verify_reason")) or None,
            has_custom_avatar=("aweme/100x100/aweme-avatar" not in
                               (dig(u,"avatar_thumb.url_list.0", default="") or "")) or None,
            post_id=c.get("cid"), thread_id=thread_id,
            created_at=datetime.fromtimestamp(c["create_time"], timezone.utc).isoformat()
                       if c.get("create_time") else None,
            text_content=c.get("text",""), like_count=c.get("digg_count"),
            reply_count=c.get("reply_comment_total"),
            is_reply=is_reply,
            is_repost=False, repost_count=None, source_device=None,
            media_types=["sticker"] if c.get("image_list") else ["text"])))
    return out

PARSERS = {"tiktok": parse_tiktok}

def inspect_payloads(platform: str, n: int = 2):
    for row in jsonl_read(RAW / f"{platform}_payloads.jsonl")[:n]:
        print("URL:", row["_url"][:110])
        print(json.dumps(row["payload"], ensure_ascii=False)[:1200], "\n---")

def build_browser_canonical(platform: str):
    rows = jsonl_read(RAW / f"{platform}_payloads.jsonl")
    if not rows:
        print(f"{platform}: no payloads captured"); return None
    recs = []
    for row in rows:
        try: recs += PARSERS[platform](row)
        except Exception as e: print(f"  ! parse error: {type(e).__name__}: {e}")
    if not recs:
        print(f"{platform}: 0 parsed — run inspect_payloads('{platform}')"); return None
    df = to_frame(recs).drop_duplicates(subset=["post_id"], keep="last")
    save_canonical(df, platform)
    print(f"{platform}: {len(df)} comments, {df.user_id.nunique()} accounts")
    session_seen = globals().get("SESSION_SEEN", {}).get(platform)
    if session_seen is not None and len(df) != session_seen:
        extra = len(df) - session_seen
        print(f"  note: this run captured {session_seen} unique comments, but "
              f"canonical/{platform}.csv has {len(df)} total ({extra:+d}) — the difference "
              f"comes from raw/{platform}_payloads.jsonl accumulating across earlier runs "
              f"(it appends, never auto-clears). Not a bug; delete that file for a clean "
              f"recollection if you want canonical/{platform}.csv to reflect only this run.")
    return df

results_browser = {p: build_browser_canonical(p)
                   for p in sorted(set(t["platform"] for t in BROWSER_TARGETS))}

def coverage(platform: str):
    rows = jsonl_read(RAW / f"{platform}_payloads.jsonl")
    if not rows:
        print(f"{platform}: nothing captured yet"); return
    totals = [dig(r["payload"], "total", default=None) for r in rows]
    totals = [t for t in totals if t is not None]
    reported_total = max(totals) if totals else None

    seen_ids = set()
    for r in rows:
        for c in dig(r["payload"], "comments", default=[]) or []:
            if c.get("cid"): seen_ids.add(c["cid"])

    print(f"{platform}: {len(seen_ids)} unique comments captured "
          f"out of {reported_total if reported_total else 'unknown'} reported by the platform")
    if reported_total:
        pct = 100 * len(seen_ids) / reported_total
        print(f"  coverage: {pct:.1f}%")
        if pct < 90:
            print("  -> re-run the browser cell on this target, or raise max_scroll_rounds")

for p in sorted(set(t["platform"] for t in BROWSER_TARGETS)):
    coverage(p)

def _extract_profile_from_page(page):
    """
    Best-effort extraction of follower/following/post counts, verification, and bio from a
    TikTok profile page's own embedded state. TikTok server-renders this into a JSON script
    tag before any client-side API call fires, which is more reliable than waiting on a
    specific XHR that may or may not happen depending on how the page was reached. Tries
    known script-tag ids and JSON shapes; TikTok has changed both before and may again — if
    this stops finding data, inspect a profile page's script tags by hand (View Source ->
    search for '__UNIVERSAL_DATA_FOR_REHYDRATION__') and adjust the paths below.

    Note: "webapp.user-detail" below is ONE dict key containing a literal dot, not a nested
    path — TikTok's __DEFAULT_SCOPE__ object really is keyed that way, so this uses direct
    dict access rather than dig() (which would split "webapp.user-detail" into two levels).
    """
    for script_id in ("__UNIVERSAL_DATA_FOR_REHYDRATION__", "SIGI_STATE"):
        try:
            raw = page.locator(f"#{script_id}").text_content(timeout=2000)
        except Exception:
            continue
        if not raw:
            continue
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(data, dict):
            continue

        default_scope = data.get("__DEFAULT_SCOPE__")
        if isinstance(default_scope, dict):
            user_info = default_scope.get("webapp.user-detail", {}).get("userInfo") \
                        if isinstance(default_scope.get("webapp.user-detail"), dict) else None
            if isinstance(user_info, dict) and (user_info.get("user") or user_info.get("stats")):
                return user_info.get("user") or {}, user_info.get("stats") or {}

        user_module = data.get("UserModule")
        if isinstance(user_module, dict):
            users_map = user_module.get("users")
            if isinstance(users_map, dict) and users_map:
                user = next(iter(users_map.values()), {}) or {}
                stats_map = user_module.get("stats") or {}
                stats = next(iter(stats_map.values()), {}) or {} if isinstance(stats_map, dict) else {}
                if user or stats:
                    return user, stats
    return {}, {}

PROFILE_FIELDS = ("followers_count", "following_count", "total_posts_count",
                   "is_verified", "bio_text")

def _load_profile_cache() -> dict:
    """Reads raw/tiktok_profiles.jsonl into {username: {...}}, last entry per username wins
    (the file is append-only, so a re-enrichment of the same account appends a newer row
    rather than replacing the old one)."""
    cache = {}
    for row in jsonl_read(RAW / "tiktok_profiles.jsonl"):
        uname = row.get("username")
        if uname:
            cache[uname] = {k: row.get(k) for k in PROFILE_FIELDS}
    return cache

def enrich_tiktok_profiles(usernames, pause_ms: int = 1500, max_accounts: int | None = None,
                            force_refresh: bool = False):
    """
    Visits each unique TikTok commenter's own profile page — reusing the SAME logged-in
    persistent browser_profile from §5, so no fresh login is needed — and pulls
    follower/following/post counts, verification, and bio, none of which the comment API
    itself returns (that's why those columns were null before). This is a public,
    one-page-per-account crawl (no login automation), but it is one extra page load per
    unique account — for hundreds of accounts that takes a while, hence `max_accounts` to
    cap a first test run before committing to enriching everyone.

    Cache-aware: usernames already present in raw/tiktok_profiles.jsonl are reused instead
    of re-visited (pass force_refresh=True to ignore the cache). This matters beyond just
    saving time — without it, a re-run that gets rate-limited on repeat profile visits
    (plausible right after a big enrichment pass) returns nothing new, and the earlier
    successful enrichment gets silently lost downstream since canonical/tiktok.csv is
    rewritten from whatever this call returns. Reusing the cache means a failed re-run
    degrades to "no new accounts added", never to "previously-enriched accounts un-enriched".
    """
    todo = sorted({u for u in usernames if u})
    if not todo:
        print("no usernames to enrich"); return {}

    cached = {} if force_refresh else _load_profile_cache()
    to_visit = [u for u in todo if force_refresh or u not in cached]
    if max_accounts:
        to_visit = to_visit[:max_accounts]

    fresh = {}
    if to_visit:
        def _run():
            pw = sync_playwright().start()
            ctx = pw.chromium.launch_persistent_context(
                user_data_dir=str(PROFILE / "tiktok"), headless=False,
                viewport={"width": 1440, "height": 900},
                locale="id-ID", timezone_id="Asia/Jakarta",
                args=["--disable-blink-features=AutomationControlled"],
            )
            page = ctx.pages[0] if ctx.pages else ctx.new_page()
            results = {}
            print(f"enriching {len(to_visit)} TikTok profile(s) "
                  f"({len(todo) - len(to_visit)} already cached, reused)...")
            for i, uname in enumerate(to_visit):
                try:
                    page.goto(f"https://www.tiktok.com/@{uname}", wait_until="domcontentloaded", timeout=30_000)
                    page.wait_for_timeout(pause_ms)
                    user, stats = _extract_profile_from_page(page)
                    if user or stats:
                        results[uname] = {
                            "followers_count": dig(stats, "followerCount", default=None),
                            "following_count": dig(stats, "followingCount", default=None),
                            "total_posts_count": dig(stats, "videoCount", default=None),
                            "is_verified": dig(user, "verified", default=None),
                            "bio_text": dig(user, "signature", default=None) or None,
                        }
                except Exception as e:
                    print(f"  ! {uname}: {type(e).__name__}: {e}")
                if (i + 1) % 25 == 0:
                    print(f"  {i+1}/{len(to_visit)} profiles visited, {len(results)} succeeded so far")
            ctx.close(); pw.stop()
            return results

        with ThreadPoolExecutor(max_workers=1) as ex:
            fresh = ex.submit(_run).result()
        if fresh:
            jsonl_append(RAW / "tiktok_profiles.jsonl",
                         [{"username": k, **v, "_collected_at": now_utc()} for k, v in fresh.items()])
    else:
        print(f"all {len(todo)} usernames already cached in raw/tiktok_profiles.jsonl — nothing to visit")

    combined = {**cached, **fresh}
    result = {u: combined[u] for u in todo if u in combined}
    print(f"profiles: {len(fresh)} newly visited + {len(result) - len(fresh)} reused from cache "
          f"= {len(result)}/{len(todo)} total")
    return result

def merge_profile_enrichment(df: pd.DataFrame, profiles: dict) -> pd.DataFrame:
    """Profile-page values are more authoritative than anything guessed from the comment
    payload (e.g. is_verified there only catches enterprise verification), so they overwrite
    the existing column where available rather than just filling nulls."""
    if not profiles or "username" not in df.columns:
        return df
    df = df.copy()
    for col in ("followers_count", "following_count", "total_posts_count", "is_verified", "bio_text"):
        enriched = df["username"].map(lambda u: profiles.get(u, {}).get(col))
        df[col] = enriched.combine_first(df[col])
    return df

if results_browser.get("tiktok") is not None:
    _tt_usernames = results_browser["tiktok"]["username"].dropna().unique().tolist()
    _tt_profiles = enrich_tiktok_profiles(_tt_usernames)
    if _tt_profiles:
        results_browser["tiktok"] = merge_profile_enrichment(results_browser["tiktok"], _tt_profiles)
        save_canonical(results_browser["tiktok"], "tiktok")
        n_have = results_browser["tiktok"]["followers_count"].notna().sum()
        print(f"tiktok: {len(_tt_profiles)} accounts profile-enriched "
              f"({n_have}/{len(results_browser['tiktok'])} comment rows now have followers_count)")

tiktok: 695 comments, 687 accounts
tiktok: 695 unique comments captured out of 5235 reported by the platform
  coverage: 13.3%
  -> re-run the browser cell on this target, or raise max_scroll_rounds


NameError: name 'ThreadPoolExecutor' is not defined

## 7. Assemble, derive features, coordination signals

In [6]:
ACCOUNT_KEY = "global_user_id"   # the identity the classifier labels: "<platform>:<user_id>"

def _coerce_bool(series: pd.Series) -> pd.Series:
    """is_reply/is_verified survive a CSV round-trip as either real bools or the strings
    'True'/'False'; a plain .astype(bool) would turn the string 'False' into True. Map
    explicitly instead."""
    truthy = {True, "True", "true", 1, "1", 1.0}
    return series.map(lambda v: v in truthy)

def derive_features(df: pd.DataFrame, window_days=None) -> pd.DataFrame:
    df = df.copy()
    if ACCOUNT_KEY not in df.columns:
        df[ACCOUNT_KEY] = df["_platform"] + ":" + df["user_id"].astype(str)
    df["created_at_dt"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)

    # observed_activity_rate: comments per day within the collection window. `span` is the
    # spread of comment timestamps across everything collected — keep the videos in one
    # topic/time window (per the multi-video plan) so this stays a meaningful rate rather
    # than count/(years).
    span = window_days
    if span is None:
        s = df["created_at_dt"].dropna()
        span = max((s.max() - s.min()).total_seconds() / 86400, 1/24) if len(s) > 1 else 1.0
    df["observed_activity_rate"] = df.groupby(ACCOUNT_KEY)["post_id"].transform("count") / span

    df["_dupe_key"] = df["text_content"].map(dedupe_key)
    valid = df["_dupe_key"] != ""
    df["duplicate_text_count"] = (
        df[valid].groupby([ACCOUNT_KEY, "_dupe_key"])["post_id"].transform("count")
    ).reindex(df.index).fillna(0).astype(int)
    df["corpus_duplicate_count"] = (
        df[valid].groupby("_dupe_key")["post_id"].transform("count")
    ).reindex(df.index).fillna(0).astype(int)
    df["distinct_users_same_text"] = (
        df[valid].groupby("_dupe_key")[ACCOUNT_KEY].transform("nunique")
    ).reindex(df.index).fillna(0).astype(int)

    df["text_length"]   = df["clean_text"].fillna("").str.len()
    df["hashtag_count"] = df["hashtags"].map(lambda v: len(v) if isinstance(v,(list,tuple)) else 0)
    df["mention_count"] = df["user_mentions"].map(lambda v: len(v) if isinstance(v,(list,tuple)) else 0)
    df["url_count"]     = df["urls"].map(lambda v: len(v) if isinstance(v,(list,tuple)) else 0)
    df["media_count"]   = df["media_types"].map(lambda v: len(v) if isinstance(v,(list,tuple)) else 0)

    # Profile-derived features. followers_count/following_count/total_posts_count/is_verified
    # are populated for YouTube via yt_enrich_channels (§4) and for TikTok via the profile
    # enrichment pass (§6, enrich_tiktok_profiles) — not every account will have these filled
    # (enrichment is best-effort / can be capped), and following_count has no YouTube
    # equivalent, so this stays genuinely null sometimes. LightGBM handles that natively;
    # don't impute it.
    followers = pd.to_numeric(df["followers_count"], errors="coerce")
    following = pd.to_numeric(df["following_count"], errors="coerce")
    df["follower_to_following_ratio"] = np.log1p(followers) / np.log1p(following).replace(0, np.nan)
    df["bio_length"] = df["bio_text"].fillna("").str.len()
    if "is_verified" in df.columns:
        df["is_verified"] = df["is_verified"].map(
            lambda v: _coerce_bool(pd.Series([v])).iloc[0] if pd.notna(v) else None)
    # has_custom_avatar is NOT used as a feature: TikTok's default-avatar URL marker
    # ("aweme/100x100/aweme-avatar") is stale — TikTok's CDN paths changed (now
    # tos-<region>-avt-.../...), so the check in parse_tiktok never matches and every TikTok
    # account comes back True regardless of its actual avatar. Non-discriminative until
    # someone inspects a real default-avatar TikTok account and updates that marker.
    # handle_digit_suffix: username ending in a long digit run (e.g. user48213957) — a cheap,
    # platform-invariant tell for auto-generated handles, common on bulk-created accounts.
    df["handle_digit_suffix"] = (
        df["username"].fillna("").astype(str).str.contains(r"\d{6,}$").astype(int)
    )

    return df.drop(columns=["_dupe_key"])

def coordination_features(df: pd.DataFrame, min_shared=2, max_group=1000) -> pd.DataFrame:
    """Thread co-occurrence (do the same accounts keep landing in the same comment threads?)
    plus median thread-arrival gap. Keyed on ACCOUNT_KEY so a TikTok and a YouTube account
    can never be conflated by a coincidentally-equal raw user_id. Repost-based coordination
    doesn't apply — TikTok/YouTube comments have no repost concept (that was X-only)."""
    ut = df[[ACCOUNT_KEY, "thread_id"]].dropna().drop_duplicates()
    counts = ut.groupby(ACCOUNT_KEY)["thread_id"].nunique()
    out = pd.DataFrame({ACCOUNT_KEY: counts.index, "thread_co_groups": counts.values})

    if not ut.empty:
        groups = ut.groupby("thread_id")[ACCOUNT_KEY].apply(list)
        pair = Counter()
        for users in groups:
            u = sorted(set(users))
            # max_group bounds the O(k^2) pairing; with per-thread grouping (not per-video)
            # threads are small, so this rarely triggers and never blows up.
            if 1 < len(u) <= max_group:
                pair.update(combinations(u, 2))
        partners, strength = {}, {}
        for (a, b), c in pair.items():
            if c < min_shared: continue
            partners.setdefault(a, set()).add(b); partners.setdefault(b, set()).add(a)
            strength[a] = max(strength.get(a, 0), c); strength[b] = max(strength.get(b, 0), c)
        out["thread_co_partners"] = out[ACCOUNT_KEY].map(lambda u: len(partners.get(u, ())))
        out["thread_co_max"]      = out[ACCOUNT_KEY].map(lambda u: strength.get(u, 0))
        out["thread_co_rate"]     = out["thread_co_max"] / out["thread_co_groups"].clip(lower=1)
    else:
        out["thread_co_partners"] = out["thread_co_max"] = out["thread_co_rate"] = 0

    d = df.dropna(subset=["thread_id", ACCOUNT_KEY]).copy()
    d["ts"] = pd.to_datetime(d["created_at"], errors="coerce", utc=True)
    d = d.dropna(subset=["ts"]).sort_values(["thread_id", "ts"])
    d["gap"] = d.groupby("thread_id")["ts"].diff().dt.total_seconds().abs()
    sync = d.groupby(ACCOUNT_KEY)["gap"].median().rename("median_thread_arrival_gap")
    return out.merge(sync, on=ACCOUNT_KEY, how="left")

# ------- Per-comment table: kept for hand-labelling / audit (has the raw text). -------
ID_COLUMNS = ["global_user_id", "user_id", "username", "display_name", "_platform", "post_id",
              "created_at", "text_content", "bio_text"]
# Per-comment model features. hashtag_count..observed_activity_rate are platform-invariant.
# followers_count..bio_length are profile-derived: real values on both platforms where
# enrichment succeeded, structurally null where it didn't run/couldn't (following_count has
# no YouTube equivalent at all) — LightGBM handles that natively.
FEATURE_COLUMNS = [
    "hashtag_count", "mention_count", "url_count", "media_count", "is_reply", "text_length",
    "language", "duplicate_text_count", "corpus_duplicate_count", "distinct_users_same_text",
    "thread_co_partners", "thread_co_rate", "median_thread_arrival_gap", "observed_activity_rate",
    "followers_count", "following_count", "total_posts_count", "follower_to_following_ratio",
    "is_verified", "bio_length", "handle_digit_suffix",
]

# ------- Account table: one row per account — the grain the classifier actually labels. -------
# Per-comment features are summarised (mean/max/fraction); account-level features
# (coordination, activity rate, profile stats) are constant per account, so we take the
# first value.
ACCOUNT_FEATURE_COLUMNS = [
    "text_length_mean", "text_length_max",
    "hashtag_count_mean", "hashtag_count_max",
    "mention_count_mean", "mention_count_max",
    "url_count_mean", "url_count_max",
    "media_count_mean", "media_count_max",
    "reply_fraction",
    "duplicate_text_count_max", "corpus_duplicate_count_max", "distinct_users_same_text_max",
    "thread_co_partners", "thread_co_rate", "median_thread_arrival_gap", "observed_activity_rate",
    "followers_count", "following_count", "total_posts_count", "follower_to_following_ratio",
    "is_verified", "bio_length", "handle_digit_suffix",
    "n_comments", "n_videos",
]

def aggregate_accounts(df: pd.DataFrame) -> pd.DataFrame:
    """Roll the per-comment table up to one row per account (ACCOUNT_KEY). Adjust the
    per-column choices below to change how a feature is summarised."""
    d = df.copy()
    d["is_reply"] = _coerce_bool(d["is_reply"])
    d["_video"] = d["_source_url"].fillna("").map(lambda u: str(u).split("?")[0])
    g = d.groupby(ACCOUNT_KEY)

    acc = pd.DataFrame(index=g.size().index)
    for col in ["text_length", "hashtag_count", "mention_count", "url_count", "media_count"]:
        acc[f"{col}_mean"] = g[col].mean()
        acc[f"{col}_max"]  = g[col].max()
    acc["reply_fraction"] = g["is_reply"].mean()
    for col in ["duplicate_text_count", "corpus_duplicate_count", "distinct_users_same_text"]:
        acc[f"{col}_max"] = g[col].max()
    for col in ["thread_co_partners", "thread_co_rate", "median_thread_arrival_gap",
                "observed_activity_rate", "followers_count", "following_count",
                "total_posts_count", "follower_to_following_ratio", "is_verified", "bio_length",
                "handle_digit_suffix",
                "_platform", "user_id", "username", "display_name"]:
        acc[col] = g[col].first()
    acc["n_comments"] = g.size()
    acc["n_videos"]   = g["_video"].nunique()
    acc["dominant_language"] = g["language"].agg(
        lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else None)
    return acc.reset_index()

def assemble(platforms=("youtube", "tiktok")) -> pd.DataFrame:
    frames = [load_canonical(p) for p in platforms if (CANONICAL / f"{p}.csv").exists()]
    if not frames:
        raise FileNotFoundError("no canonical CSV files — nothing was collected")
    df = pd.concat(frames, ignore_index=True)
    df[ACCOUNT_KEY] = df["_platform"] + ":" + df["user_id"].astype(str)
    df = derive_features(df)
    df = df.merge(coordination_features(df), on=ACCOUNT_KEY, how="left")

    # per-comment table (for labelling / audit — keeps text_content)
    comment_out = df[ID_COLUMNS + FEATURE_COLUMNS].copy()
    comment_out.to_csv(FEATURES / "dataset.csv", index=False)

    # account table (for modelling — one row per account)
    acc = aggregate_accounts(df)
    acc_cols = ["global_user_id", "_platform", "username", "display_name", "dominant_language"] \
               + ACCOUNT_FEATURE_COLUMNS
    acc = acc[[c for c in acc_cols if c in acc.columns]]
    acc.to_csv(FEATURES / "dataset_accounts.csv", index=False)

    print(f"comments: {len(comment_out)} rows, {len(FEATURE_COLUMNS)} features "
          f"-> {FEATURES / 'dataset.csv'}")
    print(f"accounts: {len(acc)} rows, {len(ACCOUNT_FEATURE_COLUMNS)} features "
          f"-> {FEATURES / 'dataset_accounts.csv'}")
    n_profiled = acc["followers_count"].notna().sum() if "followers_count" in acc.columns else 0
    print(f"platforms: {sorted(df['_platform'].dropna().unique())} | "
          f"{n_profiled}/{len(acc)} accounts have profile stats (followers/following/posts)")
    return df

df = assemble()
accounts = aggregate_accounts(df)
accounts.groupby("_platform")[["n_comments", "n_videos", "duplicate_text_count_max",
                               "thread_co_partners", "followers_count"]].mean(numeric_only=True).round(3)

comments: 1195 rows, 22 features -> buzzer_data\features\dataset.csv
accounts: 1150 rows, 28 features -> buzzer_data\features\dataset_accounts.csv
platforms: ['tiktok', 'youtube'] | 463/1150 accounts have profile stats (followers/following/posts)


,n_comments,n_videos,duplicate_text_count_max,thread_co_partners,followers_count
_platform,,,,,
tiktok,1.012,1.0,0.953,0.000,NaN
youtube,1.080,1.0,0.996,0.004,217.842


## 8. Pseudonymise and export

Keep the salt out of anything you commit or share. If you lose it you cannot re-link identities —
that is the point.

In [8]:
import getpass

PSEUDONYM_SALT = os.environ.get("PSEUDONYM_SALT") or getpass.getpass("Pseudonymisation salt: ")

def pseudonymise(frame: pd.DataFrame, salt: str) -> pd.DataFrame:
    """Hash the identity columns and drop free-text / real-name columns, for a shareable copy.
    global_user_id is hashed as a whole string, so the comment-level and account-level public
    files stay joinable on it (same salt -> same hash) without exposing the raw handle."""
    if not salt: raise ValueError("empty salt")
    frame = frame.copy()
    for c in ("global_user_id", "user_id", "username"):
        if c in frame.columns:
            frame[c] = frame[c].fillna("").map(
                lambda v: hashlib.sha256((salt + str(v)).encode()).hexdigest()[:16] if v != "" else None)
    return frame.drop(columns=["text_content", "display_name", "bio_text"], errors="ignore")

# Mirror each real output into a pseudonymised, shareable copy (read back what assemble wrote,
# so the public files always match the columns actually exported).
comments_pub = pseudonymise(pd.read_csv(FEATURES / "dataset.csv"), PSEUDONYM_SALT)
comments_pub.to_csv(FEATURES / "dataset_public.csv", index=False)

accounts_pub = pseudonymise(pd.read_csv(FEATURES / "dataset_accounts.csv"), PSEUDONYM_SALT)
accounts_pub.to_csv(FEATURES / "dataset_accounts_public.csv", index=False)

print("comment-level:", FEATURES / "dataset.csv", "(full, keeps text + bio for labelling)")
print("            ->", FEATURES / "dataset_public.csv", "(pseudonymised; text_content/display_name/bio_text dropped)")
print("account-level:", FEATURES / "dataset_accounts.csv", "(full — modelling input)")
print("            ->", FEATURES / "dataset_accounts_public.csv", "(pseudonymised)")

comment-level: buzzer_data\features\dataset.csv (full, keeps text + bio for labelling)
            -> buzzer_data\features\dataset_public.csv (pseudonymised; text_content/display_name/bio_text dropped)
account-level: buzzer_data\features\dataset_accounts.csv (full — modelling input)
            -> buzzer_data\features\dataset_accounts_public.csv (pseudonymised)


## 9. Checklist

- **TikTok profile enrichment runs automatically after parsing** (§6, `enrich_tiktok_profiles`)
  — visits each unique commenter's own profile page (reusing the already-logged-in
  `browser_profile/tiktok` session, no fresh login) to pull `followers_count`,
  `following_count`, `total_posts_count`, `is_verified`, `bio_text`, none of which the comment
  API itself returns. It's one extra page load per unique account — for hundreds of accounts
  this takes a while; pass `max_accounts=N` to `enrich_tiktok_profiles` to cap a first test run.
  Best-effort: TikTok's embedded profile-data script tag/shape has changed before and may
  again — if `n_have` in the §6 print stays 0, inspect a profile page's
  `__UNIVERSAL_DATA_FOR_REHYDRATION__` script tag by hand and adjust
  `_extract_profile_from_page`.
- **Profile-derived features are now in both feature sets**: `followers_count`,
  `following_count`, `total_posts_count`, `follower_to_following_ratio`, `is_verified`,
  `bio_length`, `handle_digit_suffix` (§7). These are real, not structurally null, on both
  platforms now — but `following_count` has no YouTube equivalent (channels don't expose a
  public following count) and TikTok enrichment is best-effort, so some stay genuinely null.
  LightGBM handles that natively; the pipeline does not impute these.
  `account_created_at`/`source_device`/`repost_count` stay excluded from both feature sets —
  those are platform-structural gaps (TikTok never exposes account-creation date; the other two
  were X-only concepts), not something more scraping fixes.
  `follower_to_following_ratio`/`posts_per_day`/`duplicate_text_count` were also removed from
  `CANONICAL_FIELDS` (§3) — they were never actually populated at canonical-write time (all
  computed downstream in `derive_features` instead), so they were just always-null clutter in
  `canonical/*.csv`, not a real per-platform gap worth documenting like the ones above.
- **`has_custom_avatar` is collected but deliberately NOT a feature.** TikTok's default-avatar
  URL marker in `parse_tiktok` (`aweme/100x100/aweme-avatar`) is stale — TikTok's CDN paths
  changed (now `tos-<region>-avt-.../...`), so the check never matches and every TikTok account
  reads as `True`. Non-discriminative until someone inspects a real default-avatar TikTok
  account and updates the marker string in `parse_tiktok` — then it can go back into
  `FEATURE_COLUMNS`/`ACCOUNT_FEATURE_COLUMNS` and `derive_features`'s coercion block.
- **`enrich_tiktok_profiles` is cache-aware** (§6) — it reuses `raw/tiktok_profiles.jsonl` for
  usernames already enriched and only visits new ones, so a re-run can't silently revert
  previously-good profile data if a repeat enrichment attempt gets rate-limited (this happened
  once: a full successful 663/687-account enrichment was overwritten by a same-session re-run
  that got throttled and returned nothing new — canonical/tiktok.csv reverted to all-null
  profile columns even though the good data was still sitting unused in the cache file). Pass
  `force_refresh=True` to intentionally re-visit everyone instead of trusting the cache.
- **Strategy: breadth over depth.** Both platforms are capped per video (`YT_MAX_PER_VIDEO`,
  `max_comments` in `BROWSER_TARGETS` — default 500) rather than scraped to exhaustion. Add more
  videos to `YT_VIDEO_IDS` / `BROWSER_TARGETS` instead of raising the caps — coordination
  features (`thread_co_*`, `corpus_duplicate_count`) only fire when the same accounts recur
  *across* videos, so more same-topic videos beats fewer bigger ones.
- **Two output grains.** `features/dataset.csv` is one row per **comment** (keeps `text_content`
  and `bio_text` for hand-labelling and audit). `features/dataset_accounts.csv` is one row per
  **account** (`global_user_id`) — this is the grain the classifier labels and trains on. Each
  has a pseudonymised `*_public.csv` sibling with identity columns hashed and text/bio/real-name
  dropped; the two public files stay joinable on the hashed `global_user_id`.
- **Account = `global_user_id` (`<platform>:<user_id>`).** All per-account aggregation and
  coordination keys on this, never on the raw `user_id`, so a TikTok and a YouTube account can't
  be conflated by a coincidentally-equal id.
- **`thread_id` = one comment thread on both platforms** (a top-level comment + its replies).
  Coordination (`thread_co_*`) and `median_thread_arrival_gap` depend on this being consistent;
  if you add a platform, make its `thread_id` follow the same rule (top-level roots its own
  thread, a reply joins its parent's).
- **Re-running is cheap.** Canonical CSVs are overwritten per platform, not appended, so a re-run
  reflects your current `BROWSER_TARGETS` / `YT_VIDEO_IDS`. Raw JSONL *does* append — delete the
  relevant `raw/*.jsonl` if you want a clean recollection.
- **TikTok scrolling always prints why it stopped** (§5): `reached max_comments cap` (expected,
  normal case now), `reached target coverage` (video was smaller than the cap), `plateaued`
  (TikTok itself stopped serving more — real throttling, not a bug), or `max_batches exhausted`
  (raise `max_batches`/`batch_rounds` if you see this before the cap is hit).
- **If your editor has the notebook open while these cells are edited, close and reopen it
  before re-running** — an open notebook keeps its own in-memory copy and can overwrite
  file-level edits on save.
- **If the browser cell captures 0 payloads**, TikTok changed its endpoint paths or comment-panel
  selector — `inspect_payloads('tiktok')` shows what came back so you can fix
  `COMMENT_ENDPOINTS` / `_hover_comment_panel` and the parser in §6.
- Ethics/ToS still apply: confirm whether BINUS requires review, keep the pseudonymisation salt
  out of anything shared, and don't automate the TikTok login or defeat its bot-detection — the
  §5 pause exists to keep this on the right side of that line. Profile enrichment is the same
  kind of public-page reading as comment scraping, not login automation.